In [0]:
import os, time
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

CATALOG = "workspace"
SCHEMA = "incident_assistent"     
LLM = "databricks-meta-llama-3-3-70b-instruct"

base = f"/Volumes/{CATALOG}/{SCHEMA}/docs"
bronze_table = f"{CATALOG}.{SCHEMA}.bronze_docs"
TABLE_A = f"{CATALOG}.{SCHEMA}.chunks_small"
TABLE_B = f"{CATALOG}.{SCHEMA}.chunks_large"
INDEX_A = f"{CATALOG}.{SCHEMA}.idx_small"
INDEX_B = f"{CATALOG}.{SCHEMA}.idx_large"
CHUNK_SCHEMA = "chunk_id string, doc_id string, chunk_no int, text string, doc_hash string"

w = WorkspaceClient()

def search(question, index_name, k=3):
    res = w.vector_search_indexes.query_index(
        index_name=index_name,
        columns=["chunk_id", "doc_id", "text"],
        query_text=question,
        num_results=k,
    )
    return res.result.data_array or []

def call_llm(prompt, max_tokens=400):
    resp = w.serving_endpoints.query(
        name=LLM,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=prompt)],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return resp.choices[0].message.content

def ask(question, index_name, k=3):
    hits = search(question, index_name, k)
    context = "\n\n".join(f"[{h[1]}] {h[2]}" for h in hits)
    prompt = f"""You are a data pipeline support assistant.
Answer ONLY using the context below.
If the answer is not in the context, say "I don't know".
At the end, list the source ids in square brackets.

Context:
{context}

Question: {question}"""
    return call_llm(prompt)

print("Runbook folder exists:", os.path.isdir(f"{base}/runbooks"))
print(spark.table(TABLE_A).count(), "small chunks |", spark.table(TABLE_B).count(), "large chunks")

Runbook folder exists: True
61 small chunks | 30 large chunks


In [0]:
def chunk_text(text, chunk_size, overlap):
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
        start += chunk_size - overlap
    return chunks

def make_rows(row, chunk_size, overlap):
    return [(f"{row['doc_id']}_{i}", row["doc_id"], i, c, row["doc_hash"])
            for i, c in enumerate(chunk_text(row["text"], chunk_size, overlap))]

In [0]:
def refresh_bronze():
    raw = (spark.read.format("text").option("wholetext", "true")
           .load(f"{base}/runbooks/*.md")
           .select(F.col("_metadata.file_path").alias("file_path"),
                   F.col("value").alias("text")))
    raw = (raw.withColumn("doc_id", F.regexp_extract("file_path", r"([^/]+)\.md$", 1))
              .withColumn("doc_hash", F.md5("text")))
    raw.write.mode("overwrite").saveAsTable(bronze_table)
    print("Bronze refreshed:", spark.table(bronze_table).count(), "runbooks")

In [0]:
def incremental_update(chunk_table, chunk_size, overlap):
    bronze = spark.table(bronze_table)
    existing = spark.table(chunk_table).select("doc_id", "doc_hash").distinct()

    # new or changed docs: this (doc_id, doc_hash) pair is not in the chunk table yet
    changed = bronze.join(existing, ["doc_id", "doc_hash"], "left_anti").collect()
    # deleted docs: in the chunk table but no longer in Bronze
    gone = [r.doc_id for r in spark.table(chunk_table).select("doc_id").distinct()
            .join(bronze.select("doc_id"), "doc_id", "left_anti").collect()]

    remove_ids = [r["doc_id"] for r in changed] + gone
    if remove_ids:
        id_list = ",".join(f"'{i}'" for i in remove_ids)
        spark.sql(f"DELETE FROM {chunk_table} WHERE doc_id IN ({id_list})")

    new_rows = []
    for r in changed:
        new_rows += make_rows(r, chunk_size, overlap)
    if new_rows:
        spark.createDataFrame(new_rows, CHUNK_SCHEMA).write.mode("append").saveAsTable(chunk_table)

    print(f"{chunk_table.split('.')[-1]}: {len(changed)} new/changed docs "
          f"({[r['doc_id'] for r in changed]}), {len(gone)} deleted docs {gone}, "
          f"{len(new_rows)} chunks added, total now {spark.table(chunk_table).count()}")

In [0]:
def sync_and_wait(index_name, chunk_table, max_minutes=10):
    w.vector_search_indexes.sync_index(index_name)
    time.sleep(45)    # give the sync time to start before checking
    target = spark.table(chunk_table).count()
    for _ in range(max_minutes * 6):
        st = w.vector_search_indexes.get_index(index_name).status
        n = getattr(st, "indexed_row_count", None)
        print("indexed rows:", n, "| target:", target, "| ready:", st.ready)
        if n == target and st.ready:
            print("Index is up to date")
            return True
        time.sleep(10)
    print("Still syncing. Wait a few minutes, then test with a search.")
    return False

In [0]:
TEST_Q = "Who should I contact if a data skew incident is not resolved after two hours?"
print("BEFORE:\n", ask(TEST_Q, INDEX_A))

BEFORE:
 You should contact the Platform Data team in the channel platform-data-help if a data skew incident is not resolved within two hours.

[source ids: [data_skew]]


In [0]:
target_file = f"{base}/runbooks/data_skew.md"
os.makedirs(f"{base}/backup", exist_ok=True)

with open(target_file) as f:
    original_text = f.read()
with open(f"{base}/backup/data_skew_original.md", "w") as f:
    f.write(original_text)

addition = ("\n\nEscalation: if the skew problem is not resolved within two hours, "
            "contact the Platform Data team in the channel platform-data-help and attach "
            "the Spark UI stage page for the slow stage.\n")
with open(target_file, "w") as f:
    f.write(addition)

print("Backup saved. Runbook words before:", len(original_text.split()),
      "| after:", len((original_text + addition).split()))

Backup saved. Runbook words before: 31 | after: 62


In [0]:
refresh_bronze()
display(spark.table(bronze_table).select("doc_id", "doc_hash").where("doc_id = 'data_skew'"))

Bronze refreshed: 31 runbooks


doc_id,doc_hash
data_skew,fa1bc4938011505bc733759441c9926a


In [0]:
incremental_update(TABLE_A, 80, 15)      # small chunks
incremental_update(TABLE_B, 200, 40)     # large chunks

chunks_small: 1 new/changed docs (['data_skew']), 0 deleted docs [], 1 chunks added, total now 61
chunks_large: 1 new/changed docs (['data_skew']), 0 deleted docs [], 1 chunks added, total now 30


In [0]:
sync_and_wait(INDEX_A, TABLE_A)
sync_and_wait(INDEX_B, TABLE_B)

indexed rows: 61 | target: 62 | ready: True
indexed rows: 61 | target: 62 | ready: True
indexed rows: 62 | target: 62 | ready: True
Index is up to date
indexed rows: 30 | target: 30 | ready: True
Index is up to date


True

In [0]:
print("AFTER:\n", ask(TEST_Q, INDEX_A))

AFTER:
 You should contact the Platform Data team in the channel platform-data-help if a data skew incident is not resolved within two hours.

[source ids: [data_skew]]


In [0]:
new_file = f"{base}/runbooks/zz_test_runbook.md"
with open(new_file, "w") as f:
    f.write("Test runbook: purple elephant incident\n\n"
            "Symptom: the nightly export job prints purple elephant in the log.\n"
            "Fix: restart the export job and check the elephant configuration flag.\n")

refresh_bronze()
incremental_update(TABLE_A, 80, 15)
sync_and_wait(INDEX_A, TABLE_A)
print([h[1] for h in search("purple elephant in the export job log", INDEX_A, 3)])

Bronze refreshed: 31 runbooks
chunks_small: 0 new/changed docs ([]), 0 deleted docs [], 0 chunks added, total now 62
indexed rows: 62 | target: 62 | ready: True
Index is up to date
['zz_test_runbook', 'adf_pipeline_failed_activity', 'data_skew']


In [0]:
os.remove(new_file)

refresh_bronze()
incremental_update(TABLE_A, 80, 15)       # should report 1 deleted doc
sync_and_wait(INDEX_A, TABLE_A)
print([h[1] for h in search("purple elephant in the export job log", INDEX_A, 3)])

Bronze refreshed: 30 runbooks
chunks_small: 0 new/changed docs ([]), 1 deleted docs ['zz_test_runbook'], 0 chunks added, total now 61
indexed rows: 62 | target: 61 | ready: True
indexed rows: 62 | target: 61 | ready: True
indexed rows: 62 | target: 61 | ready: True
indexed rows: 61 | target: 61 | ready: True
Index is up to date
['adf_pipeline_failed_activity', 'data_skew', 'late_upstream_data']


In [0]:
with open(f"{base}/backup/data_skew_original.md") as f:
    original = f.read()
with open(target_file, "w") as f:
    f.write(original)

refresh_bronze()
incremental_update(TABLE_A, 80, 15)
incremental_update(TABLE_B, 200, 40)
sync_and_wait(INDEX_A, TABLE_A)
sync_and_wait(INDEX_B, TABLE_B)
print("RESTORED:\n", ask(TEST_Q, INDEX_A))

Bronze refreshed: 30 runbooks
chunks_small: 0 new/changed docs ([]), 0 deleted docs [], 0 chunks added, total now 61
chunks_large: 0 new/changed docs ([]), 0 deleted docs [], 0 chunks added, total now 30
indexed rows: 61 | target: 61 | ready: True
Index is up to date
indexed rows: 30 | target: 30 | ready: True
Index is up to date
RESTORED:
 You should contact the Platform Data team in the channel platform-data-help if a data skew incident is not resolved within two hours.

[source ids: [data_skew]]
